# Lab 2: 工具与执行层 — Tool Execution

对应 Harness 六层架构：第 ❷ 层

## 学习目标

- 理解 **统一工具接口** 设计模式（Factory Pattern）
- 实现 validation → execution → result 的工具管道
- 让 agent 能够真正读写文件、执行命令

## 痛点回顾

在 Lab 1 中，我们的 agent 有了 `tool_use` 意图，但无法执行——模型想调用工具，却只能收到错误。

**本 Lab 的目标：给 agent 装上真正的"手脚"。**

## 对应 Claude Code 源码

- `Tool.ts` — 统一工具接口定义（约 29KB）
- `tools/BashTool/` — Bash 命令执行工具
- `tools/FileReadTool/` — 文件读取工具
- `tools/FileWriteTool/` — 文件写入工具

Claude Code 有 30+ 个工具，但它们**全部实现同一个 Tool 接口**。我们将用 Python 复刻这个设计。


---
## 第一步：环境准备

In [1]:
from utils.llm import create_harness_client, default_model
import subprocess
import os
from abc import ABC, abstractmethod

client = create_harness_client()
MODEL = default_model()

print(f"Environment ready. Model: {MODEL}")


Environment ready. Model: deepseek-reasoner


---
## 第二步：定义统一工具接口（Tool 基类）

Claude Code 中所有工具都实现同一个接口（`Tool.ts`）。核心方法：

| 方法 | 作用 |
|------|------|
| `name` | 工具名称，API 用来匹配 |
| `description` | 工具描述，让模型理解何时使用 |
| `input_schema` | JSON Schema，定义输入参数 |
| `validate()` | 校验输入是否合法 |
| `execute()` | 真正执行工具逻辑 |

这样的设计好处是：**新增工具只需实现这个接口，不需要改动 agent loop 代码。**

In [2]:
# 统一工具接口（对应 Claude Code 的 Tool.ts）

from abc import ABC, abstractmethod

class Tool(ABC):
    """所有工具的基类。对应 Claude Code 中 Tool.ts 的统一接口。"""

    @property
    @abstractmethod
    def name(self) -> str: pass

    @property
    @abstractmethod
    def description(self) -> str: pass

    @property
    @abstractmethod
    def input_schema(self) -> dict: pass

    def validate(self, tool_input: dict) -> str | None:
        """
        校验工具输入参数。
        返回 None 表示通过，返回 str 表示错误信息。
        """
        schema = self.input_schema
        required = schema.get("required", [])
        properties = schema.get("properties", {})

        # 1. 检查缺失的必填字段
        missing = [k for k in required if k not in tool_input]
        if missing:
            return f"缺少必填参数: {', '.join(missing)}"

        # 2. 检查未知字段
        if properties:
            unknown = [k for k in tool_input if k not in properties]
            if unknown:
                return f"未知参数: {', '.join(unknown)}"

        # 3. 检查类型（基础类型映射）
        type_map = {
            "string": str,
            "integer": int,
            "number": (int, float),
            "boolean": bool,
            "array": list,
            "object": dict,
        }

        errors = []
        for key, value in tool_input.items():
            prop = properties.get(key)
            if prop and "type" in prop:
                expected_type = type_map.get(prop["type"])
                if expected_type and not isinstance(value, expected_type):
                    errors.append(
                        f"参数 '{key}' 类型错误: 期望 {prop['type']}, 实际为 {type(value).__name__}"
                    )

        # 4. 检查 enum 约束
        for key, value in tool_input.items():
            prop = properties.get(key)
            if prop and "enum" in prop and value not in prop["enum"]:
                errors.append(
                    f"参数 '{key}' 值无效: '{value}' 不在允许值 {prop['enum']} 中"
                )

        if errors:
            return "; ".join(errors)

        return None  # ✅ 校验通过

    @abstractmethod
    def execute(self, tool_input: dict) -> str: pass

    def to_api_schema(self) -> dict:
        return {
            "name": self.name,
            "description": self.description,
            "input_schema": self.input_schema,
        }

---
## 第三步：实现三个核心工具

我们实现 Claude Code 中最核心的三个工具：

| 工具 | 功能 | 危险等级 |
|------|------|----------|
| `ReadFileTool` | 读取文件内容 | 低（只读） |
| `WriteFileTool` | 写入文件 | 中（有副作用） |
| `BashTool` | 执行 shell 命令 | 高（可做任何事） |

注意危险等级——这在 Lab 4（验证与安全层）中会变得很重要。


In [3]:
# Tool 1: file reads (shared implementation)

from utils.tools import ReadFileTool

print("ReadFileTool ready")


ReadFileTool ready


In [4]:
# Tool 2: file writes (shared implementation)

from utils.tools import WriteFileTool

print("WriteFileTool ready")


WriteFileTool ready


In [5]:
# Tool 3: shell execution (shared implementation)

from utils.tools import BashTool

print("BashTool ready")
print("Windows uses PowerShell by default; Linux/macOS uses the local shell.")
print("Safety checks are added in Lab 3.")


BashTool ready
Windows uses PowerShell by default; Linux/macOS uses the local shell.
Safety checks are added in Lab 3.


---
## 第四步：构建完整的 Agent Loop

现在我们把三个工具接入 Agent Loop，实现完整的 observe-think-act 循环：

```
用户输入 → Claude API (带 tools) → 收到 tool_use
    → tool_map 查找工具 → validate → execute → 收集 tool_result
    → 送回 API → 继续循环直到 stop_reason != tool_use
```

核心改动：当收到 `tool_use` 时，我们**不再返回错误，而是真正执行工具**。


In [6]:
# 注册工具
TOOLS = [ReadFileTool(), WriteFileTool(), BashTool()]

# 工具查找表：name → Tool 实例
tool_map = {t.name: t for t in TOOLS}

# 转换为 API schema
tools_schema = [t.to_api_schema() for t in TOOLS]

print(f"✓ 已注册 {len(TOOLS)} 个工具: {list(tool_map.keys())}")
tools_schema

✓ 已注册 3 个工具: ['read_file', 'write_file', 'bash']


[{'name': 'read_file',
  'description': 'Read a file from the local workspace. Relative paths are resolved from the current working directory. If the target is a directory, return a one-level directory listing.',
  'input_schema': {'type': 'object',
   'properties': {'path': {'type': 'string',
     'description': 'Path to the file or directory'},
    'file_path': {'type': 'string',
     'description': 'Alias for path; accepted for compatibility'}}}},
 {'name': 'write_file',
  'description': 'Write a file to the local workspace. Supports both relative paths and platform-native absolute paths. Prefer relative paths when working inside the current project.',
  'input_schema': {'type': 'object',
   'properties': {'file_path': {'type': 'string',
     'description': 'Path to the file to write'},
    'path': {'type': 'string',
     'description': 'Alias for file_path; accepted for compatibility'},
    'content': {'type': 'string', 'description': 'File contents'}}}},
 {'name': 'bash',
  'descr

In [7]:
def run_agent_loop(system_prompt, tools_list, user_messages, max_turns=20):
    """Agent Loop（notebook 友好版：消息列表驱动，无 input()）"""
    def _serialize_content(content_blocks):
        """将 SDK ContentBlock 序列化为纯 dict 列表"""
        result = []
        for blk in content_blocks:
            if hasattr(blk, 'type'):
                if blk.type == 'text':
                    result.append({'type': 'text', 'text': blk.text})
                elif blk.type == 'tool_use':
                    result.append({'type': 'tool_use', 'id': blk.id, 'name': blk.name, 'input': blk.input})
                else:
                    result.append({'type': blk.type})
            elif isinstance(blk, dict):
                result.append(blk)
        return result

    _tool_map = {t.name: t for t in tools_list}
    _tools_schema = [t.to_api_schema() for t in tools_list]
    messages = []
    last_text = ""
    msg_index = 0

    for turn in range(1, max_turns + 1):
        if msg_index >= len(user_messages):
            break
        user_input = user_messages[msg_index]
        msg_index += 1
        print(f"\n[轮次 {turn}] 用户: {user_input}")
        messages.append({"role": "user", "content": user_input})

        while True:
            response = client.messages.create(
                model=MODEL, max_tokens=4096, system=system_prompt,
                tools=_tools_schema, messages=messages,
            )
            messages.append({"role": "assistant", "content": _serialize_content(response.content)})
            tool_results = []
            for blk in response.content:
                if blk.type == "text":
                    last_text = blk.text
                    print(f"\nAssistant: {blk.text[:500]}")
                elif blk.type == "tool_use":
                    tool = _tool_map.get(blk.name)
                    if tool is None:
                        result, is_error = f"未知工具 {blk.name}", True
                    else:
                        validation_error = tool.validate(blk.input)
                        if validation_error:
                            result, is_error = f"参数校验失败 {validation_error}", True
                        else:
                            result, is_error = tool.execute(blk.input), False
                    print(f"  [{blk.name}] {str(blk.input)[:80]}")
                    print(f"   -> {result[:300]}")
                    tool_results.append({
                        "type": "tool_result", "tool_use_id": blk.id,
                        "content": result, **(dict(is_error=True) if is_error else {}),
                    })
            if tool_results:
                messages.append({"role": "user", "content": tool_results})
            if response.stop_reason != "tool_use":
                break

    print("--- Agent Loop 结束 ---")
    return last_text

print("run_agent_loop 就绪")


run_agent_loop 就绪


---
## 第五步：验证运行

现在让我们测试这个有工具的 agent。试试这些指令：

1. "列出当前目录的文件"
2. "创建一个 hello.py 文件，内容是 print('hello world')"
3. "运行 hello.py"
4. "读取 hello.py 的内容"

你会看到 agent 能**真正执行**这些操作了！

In [8]:
# Run the agent loop with tools

import platform

SYSTEM_PROMPT = f"""You are a coding assistant that can read files, write files, and execute shell commands.
Current OS: {platform.system()}.
Prefer relative paths and portable commands such as ls, pwd, cat, and python.
On Windows prefer PowerShell-compatible commands; on Linux/macOS prefer bash-compatible commands.
Reply in Chinese. Use the provided tools when you need to read, write, or run commands."""

print("=" * 60)
print("Lab 2: Agent Loop + Tool Execution")
print("=" * 60)

run_agent_loop(
    system_prompt=SYSTEM_PROMPT,
    tools_list=TOOLS,
    user_messages=[
        "List the files in the current directory",
        "Create a hello.py file that prints the current system date and time",
        "Run python hello.py",
        "Read the contents of hello.py",
    ],
)


Lab 2: Agent Loop + Tool Execution

[轮次 1] 用户: List the files in the current directory

Assistant: I'll list the files in the current directory for you.
  [bash] {'command': 'ls -la'}
   -> Mode   LastWriteTime      Length Name                            
----   -------------      ------ ----                            
d----- 2026/4/15 17:16:56        .ipynb_checkpoints              
d----- 2026/4/15 20:51:04        .tmp_delete_fix_2eba4e6c        
d----- 2026/4/15 15:35:51        gom

Assistant: 当前目录包含以下文件和文件夹：

**文件夹：**
- `.ipynb_checkpoints` - Jupyter notebook 检查点文件夹
- `.tmp_delete_fix_2eba4e6c` - 临时文件夹
- `gomoku` - 五子棋游戏文件夹
- `snake_game` - 贪吃蛇游戏文件夹
- `utils` - 工具文件夹

**文件：**
- `CLAUDE.md` - Claude 相关文档
- `hello.py` - Python 脚本
- `lab1_prompt_guidance.ipynb` - 实验1：提示指导
- `lab2_tool_execution.ipynb` - 实验2：工具执行
- `lab3_safety_permission.ipynb` - 实验3：安全权限
- `lab4_context_memory.ipynb` - 实验4：上下文记忆
- `lab5_planning_coordination.ipynb` - 实验5：规划协调
- `lab6_state_persistence.ipynb` - 实验6：状

'这是 `hello.py` 文件的内容。文件包含：\n\n1. **Shebang 行**：`#!/usr/bin/env python3` - 指定使用 Python 3 解释器\n2. **文档字符串**：简要说明文件功能\n3. **导入模块**：导入 Python 的 `datetime` 模块\n4. **main() 函数**：\n   - 使用 `datetime.datetime.now()` 获取当前时间\n   - 使用 `strftime()` 方法格式化时间：\n     - `%Y-%m-%d %H:%M:%S`：标准日期时间格式\n     - `%A, %B %d, %Y`：可读的完整日期格式（星期几, 月份 日期, 年份）\n     - `%I:%M:%S %p`：12小时制时间格式（带 AM/PM）\n5. **主程序入口**：`if __name__ == "__main__":` 确保脚本可以直接运行\n\n这是一个简单但功能完整的 Python 脚本，可以显示当前系统日期和时间。'

---
## 完整集成示例：工具 + 项目记忆 + Hooks

到目前为止我们实现了 Lab 1 的提示引导层和 Lab 2 的工具执行层。
下面的 cell 把它们全部集成起来：

- **CLAUDE.md 项目记忆** → 自动注入 system prompt
- **Hooks 日志钩子** → 每次工具调用自动打印日志
- **工具执行层** → 真正执行 read_file / write_file / bash


In [9]:
# === 完整集成示例：Lab 1 (提示+记忆+Hooks) + Lab 2 (工具) ===
# 直接从 utils/ 导入 Lab 1 的实现，保证代码一致性

import os, json
from utils.project_memory import ProjectMemory
from utils.hooks import HookRunner

# ❶ 项目记忆注入 system prompt
pm = ProjectMemory()
full_prompt = pm.build_system_prompt(
    "你是一个编程助手。用中文回答。", os.getcwd())
print(f"System prompt: {len(full_prompt)} 字符（含项目记忆）")

# ❶ Hooks: 日志钩子
hooks = HookRunner()

def log_pre(name, inp):
    print(f"  [Hook-Pre] 即将执行 {name}")
    return inp
def log_post(name, inp, result):
    print(f"  [Hook-Post] {name} 完成，结果长度: {len(result)}")
    return result

hooks.register_pre_tool(log_pre)
hooks.register_post_tool(log_post)

# ❷ 增强版 Agent Loop：集成 Hooks
def run_agent_loop_with_hooks(system_prompt, tools_list, user_messages, hooks_runner=None):
    def _serialize_content(content_blocks):
        """将 SDK ContentBlock 序列化为纯 dict 列表"""
        result = []
        for blk in content_blocks:
            if hasattr(blk, 'type'):
                if blk.type == 'text':
                    result.append({'type': 'text', 'text': blk.text})
                elif blk.type == 'tool_use':
                    result.append({'type': 'tool_use', 'id': blk.id, 'name': blk.name, 'input': blk.input})
                else:
                    result.append({'type': blk.type})
            elif isinstance(blk, dict):
                result.append(blk)
        return result

    _tool_map = {t.name: t for t in tools_list}
    _tools_schema = [t.to_api_schema() for t in tools_list]
    messages, msg_index = [], 0
    for turn in range(1, 20):
        if msg_index >= len(user_messages): break
        user_input = user_messages[msg_index]; msg_index += 1
        print(f"\n[轮次 {turn}] 用户: {user_input}")
        messages.append({"role": "user", "content": user_input})
        while True:
            with client.messages.stream(
                model=MODEL, max_tokens=4096, system=system_prompt,
                tools=_tools_schema, messages=messages) as stream:
            
                # 逐 token 实时输出文本部分
                print(f'\nAssistant: ', end='', flush=True)
                for text in stream.text_stream:
                    print(text, end='', flush=True)
                print()

            # 流结束后获取完整响应
            response = stream.get_final_message()
            messages.append({"role": "assistant", "content": _serialize_content(response.content)})
            tool_results = []
            for blk in response.content:
                if blk.type == "text":
                    print(f"\nAssistant: {blk.text[:500]}")
                elif blk.type == "tool_use":
                    tool = _tool_map.get(blk.name)
                    if tool is None:
                        result, is_error = f"未知工具 {blk.name}", True
                    else:
                        validation_error = tool.validate(blk.input)
                        if validation_error:
                            result, is_error = f"参数校验失败 {validation_error}", True
                        else:
                            inp = hooks_runner.run_pre_tool(blk.name, blk.input) if hooks_runner else blk.input
                            if inp is None:
                                result, is_error = "被 Hook 阻止", True
                            else:
                                result, is_error = tool.execute(inp), False
                                if hooks_runner:
                                    result = hooks_runner.run_post_tool(blk.name, inp, result)
                    print(f"  [{blk.name}] -> {result[:200]}")
                    tool_results.append({
                        "type": "tool_result", "tool_use_id": blk.id,
                        "content": result, **(dict(is_error=True) if is_error else {})})
            if tool_results:
                messages.append({"role": "user", "content": tool_results})
            if response.stop_reason != "tool_use": break
    print("\n--- 完整集成示例结束 ---")

# 运行
print("=" * 60)
print("完整集成：❶提示引导(CLAUDE.md+Hooks) + ❷工具执行")
print("=" * 60)
run_agent_loop_with_hooks(
    system_prompt=full_prompt,
    tools_list=TOOLS,
    hooks_runner=hooks,
    user_messages=["列出当前目录的文件", "删除hello.py"],
)



扫描目录: d:\agent_harness_workshop\labs
  环境信息已注入
  发现记忆文件: d:\agent_harness_workshop\labs\CLAUDE.md
  项目记忆已注入 system prompt
System prompt: 842 字符（含项目记忆）
完整集成：❶提示引导(CLAUDE.md+Hooks) + ❷工具执行

[轮次 1] 用户: 列出当前目录的文件

Assistant: 我来查看当前目录的文件和子目录。

Assistant: 我来查看当前目录的文件和子目录。
  [Hook-Pre] 即将执行 bash
  [Hook-Post] bash 完成，结果长度: 1039
  [bash] -> Mode   LastWriteTime      Length Name                            
----   -------------      ------ ----                            
d----- 2026/4/15 17:16:56        .ipynb_checkpoints              
d-

Assistant: 当前目录包含以下内容：

## 目录 (5个)
1. `.ipynb_checkpoints/` - Jupyter notebook 检查点目录
2. `.tmp_delete_fix_2eba4e6c/` - 临时目录
3. `gomoku/` - 五子棋游戏目录
4. `snake_game/` - 贪吃蛇游戏目录
5. `utils/` - 工具函数目录

## 文件 (9个)
1. `CLAUDE.md` (333B) - 项目记忆文件，包含编码规范和项目说明
2. `hello.py` (588B) - Python 脚本文件
3. `lab1_prompt_guidance.ipynb` (50.0KB) - 实验1：提示指导
4. `lab2_tool_execution.ipynb` (35.5KB) - 实验2：工具执行
5. `lab3_safety_permission.ipynb` (40.7KB) - 实验3：安全权限
6. `lab4_context_memo

---
## 小结

### 本 Lab 你学到了什么

1. **统一工具接口（Factory Pattern）** — 所有工具实现同一个 `Tool` 基类
   - `name` + `description` + `input_schema` + `validate()` + `execute()`
   - 新增工具零改动 Agent Loop

2. **工具执行管道** — tool_use → 查找 → 校验 → 执行 → tool_result → 继续循环

3. **Agent 现在能真正做事了** — 读文件、写文件、执行命令

4. **完整集成** — 工具执行层 + Lab 1 的提示引导（CLAUDE.md + Hooks）协同工作

### 痛点预告

现在试试对 agent 说：「删除当前目录所有 .py 文件」

它会**毫不犹豫地执行** `rm *.py` — 没有任何安全检查！

→ **下一个 Lab：Lab 3 验证与安全层 — 给 agent 装上安全带，拦截危险操作。**
